# Checking an SR output

An SR run returns a file.  Before anyone looks at the detail in it, three things have
to be true, and each is cheap to test.

| pass | question |
|---|---|
| structure | twice as many pixels, same bands, same data type |
| geometry | same CRS, half the pixel size, same origin |
| radiometry | mean and standard deviation and histogram shape still make sense |

The pair here is a THEOS-2 crop over Bangkok and the x2 result produced from it.  The
run converted 12-bit to 8-bit with a 2 - 99.5 % window before the network saw it, and
converted back afterwards — which is the part the third pass is looking for.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/srcheck.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'srcheck'):
    sys.modules.pop(m, None)
from srcheck import *

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
SRC = get('dataset/srcheck/input.tif', 'input.tif')
DST = get('dataset/srcheck/sr.tif', 'sr.tif')

## 1. Structure

Three numbers from the header, compared.  The pixel count has to double in both
directions; the band count and the data type have to stay put.

In [ ]:
structure(SRC, DST)

In [ ]:
show_structure(SRC, DST)

The band count did not survive: four bands went in, three came out.  That is not
automatically a fault — this product was ordered as three-band — but it is the kind of
thing that has to be noticed here rather than three steps later, when something
downstream asks for the near infrared and finds nothing.

## 2. Geometry

`gdalinfo` prints all of this.  What matters is that the SR result covers the *same
ground*: same projection, same corner, same extent — with pixels half the size.  If
the origin moves, every coordinate downstream is wrong.

In [ ]:
geometry(SRC, DST)

In [ ]:
show_geometry(SRC, DST)

0.5 m becomes 0.25 m and the footprint does not move at all — the two outlines sit
on top of each other and the origin matches to the last decimal.  The grid panels show
what doubling the pixel count means on the ground: the same 400 m square, cut four
times as finely.

## 3. Radiometry

SR should sharpen an image, not rescale it.  So the mean should barely move, the
standard deviation should rise a little — that is the added detail — and the histogram
should keep its shape.

In [ ]:
radiometry(SRC, DST)

In [ ]:
show_radiometry(SRC, DST)

The mean moved by exactly -0.50 levels in every band, and the standard deviation rose
2 - 3 %.  Both are what a working SR run looks like.

The ends are a different story.  The brightest pixel in the red band was 4095 going in
and 2250 coming out, and the blue histogram has a tail past 2500 that the orange one
does not.  Those pixels sat outside the 2 - 99.5 % window, so they were flattened when
the data was squeezed into 8 bits, and converting back could not invent them again.
The window that made the run possible is the same window that cost the extremes.